In [0]:
# Databricks notebook source
%pip install gspread==6.2.1 google-auth==2.53.0

# COMMAND ----------

%restart_python

# COMMAND ----------

# ============================================================================
# Bronze ingestion: Happy Family Homestay bookings
#
# Source : Google Sheet (Google Form responses), read with a service account
#          stored in a Databricks secret scope.
#          Fallback: sample CSV in the Git folder if the credentials are unavailable.
# Target : CSV file in a Unity Catalog Volume (landing zone)
#          -> loaded into the bronze Delta table by 01b_bronze_load.sql
# ============================================================================

In [0]:
import json
import os

import gspread
import pandas as pd

In [0]:
# ---------- Configuration ----------
SHEET_NAME = "Happy Family Homestay - Konfirmasi Booking (Responses)"
SECRET_SCOPE = "gcp_credentials"
SECRET_KEY = "service_acc_json"

# Notebook runs from notebooks/ inside the Git folder, so data/ is one level up
FALLBACK_CSV = "../data/sample_bookings.csv"

CATALOG = "happy_family_homestay"
SCHEMA = "default"
VOLUME = "homestay_landing"
LANDING_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/homestay_raw.csv"

SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive",
]

In [0]:
# COMMAND ----------

# ---------- Setup ----------
def setup_landing_zone() -> None:
    """Create the catalog and volume for the landing file if they do not exist."""
    spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")

In [0]:
# ---------- Extract ----------
def get_google_sheets_client() -> gspread.Client:
    """Authenticate with the service account stored in a Databricks secret."""
    info = json.loads(dbutils.secrets.get(scope=SECRET_SCOPE, key=SECRET_KEY))
    return gspread.service_account_from_dict(info, scopes=SCOPES)

In [0]:
def extract_bookings() -> pd.DataFrame:
    """Read bookings from the Google Sheet, or from the sample CSV if credentials are unavailable."""
    try:
        client = get_google_sheets_client()
    except Exception as e:
        print(f"Secret scope '{SECRET_SCOPE}' unavailable ({type(e).__name__}). Using '{FALLBACK_CSV}'.")
        if not os.path.exists(FALLBACK_CSV):
            raise FileNotFoundError(
                f"No credentials and no fallback file at '{FALLBACK_CSV}' "
                f"(current folder: {os.getcwd()})."
            )
        df = pd.read_csv(FALLBACK_CSV, dtype=str)
        print(f"Loaded {len(df)} records from the Git folder.")
        return df

    records = client.open(SHEET_NAME).sheet1.get_all_records()
    df = pd.DataFrame(records)
    print(f"Extracted {len(df)} records from Google Sheets.")
    return df

In [0]:
# ---------- Load ----------
def land_to_volume(df: pd.DataFrame) -> None:
    """Write the bookings as a CSV file into the landing volume."""
    df.to_csv(LANDING_PATH, index=False)
    print(f"Landed {len(df)} records at '{LANDING_PATH}'.")

In [0]:
# COMMAND ----------

# ---------- Run ----------
setup_landing_zone()
land_to_volume(extract_bookings())